# Table of Contents

1. Pulling the Data](
1. Preparing the Data
1. Further Cleaning

# Pulling the Data

All of the data collected from others was collected

In [1]:
# !pip install gspread pandas google-auth
import gspread
from google.oauth2.service_account import Credentials
import os, pandas as pd
from pathlib import Path


/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
os.environ["GOOGLE_SHEETS_CREDENTIALS"] = (
    "/opt/notebooks/psalms_nlp_sp26/private/psalms-blind-scoring-da42a38adf3c.json"
)

os.getcwd()

'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [3]:
# Current notebook directory
notebook_dir = Path.cwd()

# Build path to the JSON
cred_path = notebook_dir.parent / "private" / "psalms-blind-scoring-da42a38adf3c.json"
print("Credential path exists?", cred_path.exists())

os.getcwd()

Credential path exists? True


'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [4]:
import socket
socket.gethostbyname("oauth2.googleapis.com")


'192.178.155.95'

In [5]:
creds = Credentials.from_service_account_file(
    os.environ["GOOGLE_SHEETS_CREDENTIALS"],
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
)

client = gspread.authorize(creds)

sheet = client.open("results_scored")

In [6]:
# Access second sheet (index 1)
worksheet2 = sheet.get_worksheet(1)

# Get all values
data = worksheet2.get_all_values()

# Convert to DataFrame (first row as header)
df = pd.DataFrame(data[1:], columns=data[0])

# storing the collected data for reference later
df.to_csv("../data/results_scored.csv", index=False, mode="w")


In [7]:
df

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3,,
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,9,5,p06,1,p03,3,p08,,
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an ode by David You hear...,6,1,p03,8,p04,7,p06,,
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning understanding Blessed are ...,3,7,p06,1,p03,3,p10,,
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,10,7,p06,6,p05,5,p09,,
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,7,2,p01,0,p03,2,p08,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,Verses where the psalmist remembers past deliv...,TFIDF,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,3,10,p06,8,p09,,,,
234,Verses where the psalmist remembers past deliv...,TFIDF,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",8,2,p10,7,p05,2,p06,,
235,Verses where the psalmist remembers past deliv...,TFIDF,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",5,7,p01,0,p06,3,p04,,
236,,,,,,,,,,,,,,,


# Preparing the data
I want each row to hold one of the four possible scored for each of the `236 results`. So if rebuilt properly, we should end up with a total of: 
$$236 * 4 = 944\ results$$

I need to start by unpivoting my own score separte from the other scores.

## Numbering the Results 
I also want to be able to reference the order of the results within each search. I collected the top 5 results from each search. There was a bug in my code that took the top 6 results from searches. I am going to just worry about the top 5 results to keep everything fair. 

In [8]:
# temporary dataframe to not break the original 
temp = df.copy()

# aqdding a column to number the indivudal results
temp['numbered_result'] = pd.NA
#temp

In [9]:
n = temp.shape[0]

# starting with the number 1 result of a query
num_result = 1

query = temp["Query"].iloc[0]
method = temp["Method"].iloc[0]

for i in range(n):
    # checking if we are in the same group fo data to be numbered
    if query == temp["Query"].iloc[i] and method == temp["Method"].iloc[i]:
        temp["numbered_result"].iloc[i] = num_result
        num_result += 1
        
    # in an new group of results
    else:
        # the current result is the number result for the new set of results
        temp["numbered_result"].iloc[i] = 1
        # reset the number result
        num_result = 2
        # update to the new target query & method
        query = temp["Query"].iloc[i]
        method = temp["Method"].iloc[i]
        
# temp.tail(20)


/tmp/ipykernel_342/2796122015.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  temp["numbered_result"].iloc[i] = num_result
/tmp/ipykernel_342/2796122015.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!


In [10]:
# select first 7 columns + last column
cols_to_keep = list(temp.columns[:7]) + [temp.columns[-1]]
caden = temp[cols_to_keep]

# caden.head()


In [11]:
caden['User'] = 'caden'

caden = caden.rename(columns={"CadenScore": "Score"})

# caden

/tmp/ipykernel_342/2631933811.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  caden['User'] = 'caden'


In [12]:
caden  = caden[['Query', 'Method', "numbered_result", 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User', 'Score']]
caden

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...
233,Verses where the psalmist remembers past deliv...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,caden,3
234,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",caden,8
235,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",caden,5
236,,,1,,,,,caden,


Moving on to prepaering the external scores.

In [13]:
external = temp[['Query', 'Method', "numbered_result", 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User1', 'Score1', 'User2', 'Score2', 'User3', 'Score3']]

#external

In [14]:
# Unpivot User/Score pairs
df_long = pd.wide_to_long(
    external,
    stubnames=["User", "Score"],  # the base column names
    i=["Query", "Method", "numbered_result", "Similarity Score (%)", "Text", "Psalm Num", "Verse"],  # columns to keep
    j="Pair",  # new column for the pair number
    sep=""      # number comes directly after the stub name
).reset_index()

# Optional: reorder columns
df_long = df_long[["Query", "Method", "numbered_result",  "Similarity Score (%)", "Text", "Psalm Num", "Verse", "Pair", "User", "Score"]]



In [15]:
external = df_long[["Query", "Method", "numbered_result",  "Similarity Score (%)", "Text", "Psalm Num", "Verse", "User", "Score"]]

external

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p06,5
1,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p03,1
2,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p08,3
3,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p03,1
4,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p04,8
...,...,...,...,...,...,...,...,...,...
709,,,1,,,,,,
710,,,1,,,,,,
711,,,2,,,,,,5.199152542
712,,,2,,,,,,5.398305085


In [16]:
#caden.head()

## Combining the Prepared Data back together

In [17]:
scores = pd.concat([caden, external], ignore_index=True)
scores

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...
947,,,1,,,,,,
948,,,1,,,,,,
949,,,2,,,,,,5.199152542
950,,,2,,,,,,5.398305085


In [18]:
(scores["Score"].notna() & (scores["Score"] != "")).sum()

945

In [19]:
#scores["Query"].unique()

We now have our intended 944 rows of data we can move on to do last few preperations for the analysis. 

# Further Data Cleaning <a href="further_cleaning"></a>

I now need to work on some of the anaiysis of the data and alot of the imediate processiong was done within another notebook so it will be coppied into here. 

## Query Categoization

In [20]:
query_categories = {
    "mercy": "Simple Keyword Queries",
    "prayer":"Simple Keyword Queries",
    "The Lord is my shepherd": "Phrase/Exact Match Queries",
    "Create in me a clean heart":"Phrase/Exact Match Queries",
    "protection from enemies": "Thematic/Semantic Queries",
    "praise in times of suffering": "Thematic/Semantic Queries",
    "How does the psalmist express trust in God while surrounded by fear and uncertainty?":
        "Long/Complex Queries",
    "Verses where the psalmist remembers past deliverance and uses it to find hope in present trials.":
        "Long/Complex Queries",
    "Rejoice, O ye heavens, sound the trumpets, ye foundation of the earth, thunder forth gladness, O ye mountains: for behold, Emanuel to the Cross our sins, and the Giver of Life hath slain death, raising up Adam; for He loveth mankind.":
        "Orthodox Service Quotes",
    "Have mercy on me, O God, have mercy on me. For my soul trusts in Thee, and in the shadow of Thy wings will I hope, until iniquity pass away.":
        "Orthodox Service Quotes",
    "For the Peace of the world": "Orthodox Service Quotes"

}

In [21]:
scores["Query Category"] = scores["Query"].map(query_categories)
#scores

In [22]:
# reordering the columns of the dataframe
scores = scores [["Query", "Query Category", "Method", "numbered_result", "Similarity Score (%)", "Text", "Psalm Num", 
                  "Verse", "User", "Score" ]]

#scores

In [23]:
# reordering the columns of the dataframe
scores = scores [["Query", "Query Category", "Method","Similarity Score (%)", "numbered_result",
                  "Text", "Psalm Num", "Verse", "User", "Score" ]]

#scores

In [24]:
# filtering to only be studying the top 5 results from each query
scores = scores[scores['numbered_result'] != 6]


pd.set_option("display.max_rows", 50)

#scores[scores['Method'] == 'TFIDF']

In [25]:
external = external[external['numbered_result'] != 6]

---

# Analysis

## Inner-Annotator Agreement
Every person that contributed to the scoring of the results aproached them differently even though the same intetion was behind each score conceived. Each result of the *236 results*, was scored up to three times. This may cause discrepency in how each result was scored overall. **Inner-Anotator Agreement** works at trying to normalize the differeint in scored overall, and for each indivual result. 
There are a few different metricsx that handel this. The data for this study is ordinal which means that a `1` is closer to `2` than `5`, making this just just a category. For this reason **Krippendorff's** alpha agreement is what's going to be used. 

The overall metric consists of the following equation:
$$
\alpha = 1 - \frac{D_0}{D_e}
$$

Where $D_0$ is:
$$
D_o = \frac{1}{n} \sum_{c} \sum_{k} o_{ck} \, \delta_{ck}^2
$$

And $D_e$ is:
$$
D_e = \frac{1}{n(n-1)} \sum_{c} \sum_{n_c} n_c * n_{k\ ordinal} \, \delta_{ck}^2
$$

In [26]:
import pandas as pd
import numpy as np

Code for: $$D_o = \frac{1}{n} \sum_{c} \sum_{k} o_{ck} \, \delta_{ck}^2$$

In [27]:
import numpy as np

o_ck = [[0.0]*11 for _ in range(11)]
pairable_n = 0

o_ck

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]

Building the $o_ck$ portion of $D_0$:

In [28]:
pairable_n = 0

# coincidence matrix (0–10 scale)
o_ck = [[0.0]*11 for _ in range(11)]

# marginal frequencies n_c
counts = [0]*11

In [29]:
def process_row_external(row):

    scores = [row['Score1'], row['Score2'], row['Score3']]

    valid_scores = []
    for s in scores:
        if not pd.isna(s) and s != "":
            val = int(float(s))
            valid_scores.append(val)
            counts[val] += 1   # <-- THIS is how you get counts

    m_u = len(valid_scores)

    if m_u < 2:
        return

    global pairable_n
    pairable_n += m_u

    for i in range(m_u):
        for j in range(m_u):
            if i == j:
                continue
            c = valid_scores[i]
            k = valid_scores[j]
            o_ck[c][k] += 1/(m_u-1)

In [30]:
for _, row in df.iterrows():
    process_row_external(row)

print(counts)

[54, 84, 58, 53, 42, 59, 57, 49, 88, 81, 84]


<div style=" padding:10px; display:inline-block;">
$$
\delta^2_{ck} = \left( \sum_{g=c}^{\max(c,g)} n_g - \frac{n_c + n_k}{2} \right)^2
$$
</div>

In [31]:
def ordinal_delta_sq(c, k):
    if c == k:
        return 0.0
    
    low = min(c, k)
    high = max(c, k)

    # cumulative counts between ranks
    cumu = sum(counts[g] for g in range(low, high+1))

    # subtract half endpoints
    cumu -= (counts[c] + counts[k]) / 2

    return cumu ** 2

In [32]:
def compute_D_o():
    total_coincidences = sum(sum(row) for row in o_ck)
    if total_coincidences == 0:
        return None  # or 0

    total = 0
    for c in range(len(o_ck)):
        for k in range(len(o_ck)):
            delta_sq = ordinal_delta_sq(c, k)
            total += o_ck[c][k] * delta_sq

    return total / total_coincidences

In [33]:
pairable_n = 0
o_ck = [[0.0]*11 for _ in range(11)]
counts = [0]*11  # important if using ordinal_delta_sq

for idx, row in df.iterrows():
    process_row_external(row)

D_o = compute_D_o()

D_o

56831.71509167842

    d_0(temp.iloc[0]
### Building the Coincidences Matrix

In [34]:
o_ck

[[10.0, 9.5, 8.0, 4.5, 2.5, 3.0, 5.0, 3.5, 3.5, 4.5, 0.0],
 [9.5, 10.0, 10.0, 5.5, 6.0, 6.0, 5.0, 6.5, 14.0, 8.0, 3.5],
 [8.0, 10.0, 6.0, 6.5, 3.5, 5.0, 3.5, 4.5, 4.0, 4.0, 3.0],
 [4.5, 5.5, 6.5, 3.0, 5.0, 4.5, 6.5, 3.5, 7.0, 4.0, 3.0],
 [2.5, 6.0, 3.5, 5.0, 1.0, 7.0, 4.0, 2.0, 5.5, 3.0, 2.5],
 [3.0, 6.0, 5.0, 4.5, 7.0, 8.0, 4.5, 5.0, 5.0, 6.0, 5.0],
 [5.0, 5.0, 3.5, 6.5, 4.0, 4.5, 4.0, 5.0, 7.0, 7.5, 5.0],
 [3.5, 6.5, 4.5, 3.5, 2.0, 5.0, 5.0, 6.0, 2.0, 4.5, 6.5],
 [3.5, 14.0, 4.0, 7.0, 5.5, 5.0, 7.0, 2.0, 11.0, 11.5, 17.5],
 [4.5, 8.0, 4.0, 4.0, 3.0, 6.0, 7.5, 4.5, 11.5, 14.0, 14.0],
 [0.0, 3.5, 3.0, 3.0, 2.5, 5.0, 5.0, 6.5, 17.5, 14.0, 24.0]]

### $D_e$
Code for: $D_e = \frac{1}{n(n-1)} \sum_{c} \sum_{n_c} n_c * n_{k\ metric} \, \delta_{ck}^2$

*Where*:
> - $n_c$ = number of times score `c` occurs in the dataset  
> - $n_k$ = number of times score `k` occurs in the dataset  
> - $\delta_{ck}^2$ = squared distance between scores `c` and `k`  
> - `n(n-1)` = total number of pairs in the dataset

Code for 
$$
D_e = \frac{\sum_c \sum_k n_c n_k \left( \sum_{g=\min(c,k)}^{\max(c,k)} n_g - \frac{n_c + n_k}{2} \right)^2}{n(n-1)}, 
\quad n = \sum_c n_c
$$

In [35]:
def compute_D_e():

    n = sum(counts)

    total = 0
    for c in range(11):
        for k in range(11):
            delta_sq = ordinal_delta_sq(c, k)
            total += counts[c] * counts[k] * delta_sq

    return total / (n*(n-1))

Now we can refer to the original formula: $\alpha = 1 - \frac{D_0}{D_e}$

In [36]:
def alpha():
    d_o = compute_D_o()
    d_e = compute_D_e()
    print("D_e = " + str(d_e))
    return 1 - (d_o / d_e)

In [37]:
external.head()

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p06,5
1,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p03,1
2,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p08,3
3,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p03,1
4,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p04,8


In [38]:
alpha()

D_e = 83078.80084745762


0.31593000245600456

- `02/20/2026`- $\alpha = 0.614410708025859$
    - I was not properly computing $D_e$ correctly. I was using $(c-k)^2$ rather than $\delta^2_{ck}$. The code above represents the ordinal computations.
- `02/21/2026`- $\alpha = 0.9999895570896626$
Debugging

## Testing with the offical python package


Confirming my results with the offical pacakge before going further. 

In [39]:
# %pip install krippendorff
import krippendorff

In [40]:
import numpy as np
import pandas as pd

temp = df

cols_without = ['Score1', 'Score2', 'Score3']
cols_with = ['CadenScore', 'Score1', 'Score2', 'Score3']

temp[cols_without] = temp[cols_without].apply(pd.to_numeric, errors='coerce')
temp[cols_with] = temp[cols_with].apply(pd.to_numeric, errors='coerce')

In [41]:
data_without_caden = temp[['Score1', 'Score2', 'Score3']].T.to_numpy()

alpha_without = krippendorff.alpha(
    reliability_data=data_without_caden,
    level_of_measurement='ordinal'
)

print("Without Caden:", alpha_without)

Without Caden: 0.3158573343580152


# Rebuilding

Resuilding the metric computation to re compute the score to be able to compute quickly for specfric sets of the data. 

In [43]:
def build_coincidence_matrix(df, score_columns, max_score=10):

    counts = [0]*(max_score+1)
    o_ck = np.zeros((max_score+1, max_score+1))
    pairable_n = 0

    for _, row in df.iterrows():
        scores = [row[col] for col in score_columns]
        
        valid_scores = []
        for s in scores:
            if not pd.isna(s) and s != "":
                val = int(float(s))
                valid_scores.append(val)
                counts[val] += 1

        m_u = len(valid_scores)
        if m_u < 2:
            continue

        pairable_n += m_u

        for i in range(m_u):
            for j in range(m_u):
                if i == j:
                    continue
                c = valid_scores[i]
                k = valid_scores[j]
                o_ck[c][k] += 1/(m_u-1)

    return counts, o_ck, pairable_n

In [44]:
def ordinal_delta_sq(c, k, counts):
    if c == k:
        return 0.0

    low = min(c, k)
    high = max(c, k)

    cumu = sum(counts[g] for g in range(low, high+1))
    cumu -= (counts[c] + counts[k]) / 2

    return cumu ** 2

In [45]:
def compute_D_o(o_ck, counts, max_score=10):

    total_coincidences = o_ck.sum()
    if total_coincidences == 0:
        return None

    total = 0
    for c in range(max_score+1):
        for k in range(max_score+1):
            delta_sq = ordinal_delta_sq(c, k, counts)
            total += o_ck[c][k] * delta_sq

    return total / total_coincidences

In [46]:
def compute_D_e(counts, max_score=10):

    n_total = sum(counts)
    if n_total < 2:
        return None

    total = 0
    for c in range(max_score+1):
        for k in range(max_score+1):
            delta_sq = ordinal_delta_sq(c, k, counts)
            total += counts[c] * counts[k] * delta_sq

    return total / (n_total * (n_total - 1))

In [47]:
def compute_alpha(D_o, D_e):

    if D_o is None or D_e is None:
        return None

    if D_e == 0:
        return 1.0

    return 1 - (D_o / D_e)

In [48]:
def krippendorff_alpha_ordinal(df, score_columns, max_score=10):

    counts, o_ck, pairable_n = build_coincidence_matrix(
        df, score_columns, max_score
    )

    D_o = compute_D_o(o_ck, counts, max_score)
    D_e = compute_D_e(counts, max_score)

    alpha = compute_alpha(D_o, D_e)

    return alpha #, D_o, D_e, counts, o_ck

With these functions built, I want to veriofy they are working the way they did before. 

In [50]:
krippendorff_alpha_ordinal(df, ['Score1', 'Score2', 'Score3'], max_score=10)

0.31593000245600456

We are getting the same score a before. This is an easier way of applying the metic because it is easier to change what data to compute. Lets. work on trying different parts of the data. 

### Adding my score

In [246]:
krippendorff_alpha_ordinal(df, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)

0.27496365676670687

### Looking at each indivual Algorithm
#### TFIDF

In [51]:
temp = df[df['Method'] == 'TFIDF']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.3590222319223847
With Caden:  0.2746219308416913


#### TFIDF x GLoVE

In [52]:
temp = df[df['Method'] == 'TFIDF_GLoVe']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.42123841220762215
With Caden:  0.39983513136824533


#### BERT

In [53]:
temp = df[df['Method'] == 'BERT']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.21139792268807145
With Caden:  0.17950681653424738


#### SBERT

In [ ]:
temp = df[df['Method'] == 'SBERT']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


## Investigating the Low Alpha Score

The overall `Krippendorff Aplpha` score I calculated was
$$\alpha_{w/out\ Caden} = 0.3132363233535458$$
and 
$$\alpha_{w/ \ Caden} = 0.27457889178612493$$

These tell us that the scores between everyone is not reliable, Lets see what might be contributing to that. 

### Looking at the everage score of each user

In [ ]:
caden

In [ ]:
import pandas as pd

# Convert 'Score' to numeric, invalid parsing becomes NaN
scores['Score'] = pd.to_numeric(scores['Score'], errors='coerce')

# Optional: drop rows where conversion failed
scores = scores.dropna(subset=['Score'])

# Finally, convert to integer
scores['Score'] = scores['Score'].astype(int)

scores['Score'].count()

In [ ]:
scores.pivot_table(index='User', values='Score', aggfunc='mean')

In [ ]:
scores['Score'].mean()

In [ ]:
import numpy as np

def score_category(score):
    # If score is missing or not a number, return np.nan
    if pd.isna(score) or score == '':
        return np.nan
    # Otherwise convert to float and categorize
    score = float(score)
    if score <= 3:
        return 1  # Low
    elif score <= 7:
        return 2  # Medium
    else:
        return 3  # High

In [ ]:
df['Score_Caden Category'] = df['CadenScore'].apply(score_category)

df['Score_1 Category'] = df['Score1'].apply(score_category)
df['Score_2 Category'] = df['Score2'].apply(score_category)
df['Score_3 Category'] = df['Score3'].apply(score_category)

In [ ]:
df.head()

### Using Krippendorff ALpha for categorical scores

In [ ]:
print(df.columns.tolist())

In [ ]:
df

In [ ]:
# Step 2: Convert to numpy array
data_without_caden = df[['Score_1 Category','Score_2 Category','Score_3 Category']].T.to_numpy()

# Step 3: Compute Krippendorff (with missing values preserved)
alpha_without = krippendorff.alpha(
    reliability_data=data_without_caden,
    level_of_measurement='ordinal'
)

print("Without Caden:", alpha_without)

From a different studying Using `Krippendorff's alpha`: 

**"A Validated Scoring Rubric for Explain-in-Plain-English Questions" <br>**
> "we first applied z-score standardization on a per question basis to account for the variations in question difficulty. After standardization, we computed the average z-score for each student to account for the fact that some students answered fewer code reading questions. "
>
- There is variation in every rater's view of the Psalms in addition to the different interpretations of each query
- No one was given an example of this because it up to interpretation as everyone is seeking the Psalsm with different prespectives, as I make note of within the introduction of the Poster and Paper. 
- Applying the `Z-score` standarization may help in these realistic short comings. 

$$
z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

$$
\mu_j = \frac{1}{n} \sum_{i=1}^{n} x_{ij}
$$

$$
\sigma_j = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_{ij} - \mu_j)^2}
$$

After the z-score is implented the data turns into interval data

In [ ]:
# Keep only numeric scores
raters = ['CadenScore', 'Score1', 'Score2', 'Score3']

# Z-score standardization (per rater)
df_z = (df[raters] - df[raters].mean()) / df[raters].std()

# Convert to numpy for Krippendorff
import krippendorff
data = df_z.T.to_numpy()

alpha = krippendorff.alpha(reliability_data=data, level_of_measurement='interval')

print("Krippendorff alpha (z-score standardized):", alpha)

Krippendorff’s alpha (ordinal) looks at how raters rank or differentiate items relative to each other.

If raters disagree on the ordering of items, subtracting each rater’s mean doesn’t help — it only shifts the scores up or down.

---

In [ ]:
from sklearn.metrics import cohen_kappa_score
import numpy as np

temp = df[df['Method'] == 'TFIDF_GLoVe']

def weighted_kappa_average(dataframe, columns):

    kappas = []

    for i in range(len(columns)):
        for j in range(i+1, len(columns)):

            r1 = dataframe[columns[i]]
            r2 = dataframe[columns[j]]

            # Remove missing values
            valid = ~(r1.isna() | r2.isna())

            kappa = cohen_kappa_score(
                r1[valid],
                r2[valid],
                weights='quadratic'
            )

            kappas.append(kappa)

    return np.mean(kappas)

In [ ]:
scores.head()

In [ ]:
pd.pivot_table(data=scores, index=['User', "Method"], values='Score', aggfunc = 'count')

In [ ]:
scores

In [ ]:
print(scores.columns.tolist())

In [ ]:
scores_piv = scores.pivot_table(
    index=['Query', 'Query Category', 'Method', 'Similarity Score (%)', 'numbered_result', 'Text', 'Psalm Num', 'Verse'],  # or whatever defines a unique row
    columns="User",
    values="Score",
    aggfunc="first"
).reset_index()

In [ ]:
scores_piv

In [ ]:
# Count how many scores each user has
counts = scores_piv.count()

# Keep users with 125 or more scores
pivoted = scores_piv.loc[:, counts >= 15]

pivoted

In [ ]:
raters = scores_piv.columns.difference([
    'Query',
    'Query Category',
    'Method',
    'Similarity Score (%)',
    'numbered_result',
    'Text',
    'Psalm Num',
    'Verse'
])

df_raters = scores_piv[raters]

In [ ]:
df_raters


In [ ]:
data = df_raters.to_numpy().T

data

In [ ]:
import pandas as pd
import krippendorff

def compute_krippendorff(df_raters, users_to_include, 
                         min_ratings_per_item=2, level='ordinal', zscore=False):
    """
    Compute Krippendorff alpha for a subset of raters, optionally z-score standardizing.

    Parameters:
    -----------
    df_raters : pd.DataFrame
        DataFrame with raters as columns and items as rows
    users_to_include : list of str
        List of rater column names to include
    min_ratings_per_item : int
        Minimum number of non-missing ratings per item to keep
    level : str
        Level of measurement ('ordinal', 'interval', etc.)
    zscore : bool
        If True, standardize each rater's scores using z-score

    Returns:
    --------
    alpha : float
        Krippendorff alpha value
    df_clean : pd.DataFrame
        Filtered DataFrame used to compute alpha
    included_users : list
        List of users actually included
    """

    # Keep only users that exist in the DataFrame
    included_users = [u for u in users_to_include if u in df_raters.columns]
    if not included_users:
        raise ValueError("No valid users to include in df_raters.")

    # Subset the DataFrame
    df_subset = df_raters[included_users].copy()

    # Apply z-score per rater if requested
    if zscore:
        df_subset = (df_subset - df_subset.mean()) / df_subset.std(ddof=0)

    # Keep only rows/items with enough ratings
    df_clean = df_subset[df_subset.notna().sum(axis=1) >= min_ratings_per_item]

    # Convert to raters × items numpy array
    data = df_clean.to_numpy().T

    # Compute Krippendorff alpha
    alpha = krippendorff.alpha(reliability_data=data, level_of_measurement=level)

    return alpha, df_clean, included_users

In [ ]:
avg_filtered = avg[
    (avg[('mean', 'Score')] > 4) & (avg[('mean', 'Score')] < 8) & (avg[('count', 'Score')] > 5)
]

avg_filtered

# Suppose avg_filtered contains your filtered users
cols_to_keep = avg_filtered.index.tolist()

# Compute alpha with z-score standardization
alpha_z, df_used_z, users_included_z = compute_krippendorff(
    df_raters, 
    cols_to_keep, 
    min_ratings_per_item=2, 
    level='interval',  # interval is suitable after z-scoring
    zscore=True
)

print("Users included:", users_included_z)
print("Number of items considered:", df_used_z.shape[0])
print("Krippendorff alpha (z-score standardized):", alpha_z)

In [ ]:
avg

In [ ]:
import pandas as pd

# Define thresholds to test
mean_thresholds = [(low, high) for low in range(1, 8) for high in range(low+1, 11)]
count_thresholds = [5, 10, 25, 50, 75, 100]

# List to store results
results = []

for mean_low, mean_high in mean_thresholds:
    for count_min in count_thresholds:
        # Filter avg table
        avg_filtered = avg[
            (avg[('mean', 'Score')] > mean_low) &
            (avg[('mean', 'Score')] < mean_high) &
            (avg[('count', 'Score')] >= count_min)
        ]
        
        # Users to include
        cols_to_keep = avg_filtered.index.tolist()
        
        # Skip if no users left
        if not cols_to_keep:
            continue
        
        for z in [True, False]:  # Loop over z-score standardization
            try:
                alpha, df_used, users_included = compute_krippendorff(
                    df_raters,
                    cols_to_keep,
                    min_ratings_per_item=2,
                    level='interval' if z else 'ordinal',
                    zscore=z
                )
            except Exception:
                alpha = None
                df_used = pd.DataFrame()
                users_included = []

            # Record results
            results.append({
                'mean_low': mean_low,
                'mean_high': mean_high,
                'count_min': count_min,
                'num_users': len(users_included),
                'num_items': df_used.shape[0],
                'zscore': z,
                'kripp_alpha': alpha
            })

# Convert results to DataFrame
alpha_results_df = pd.DataFrame(results)

# Sort by alpha descending
alpha_results_df = alpha_results_df.sort_values(by='kripp_alpha', ascending=False).reset_index(drop=True)

# Show top 10 results
alpha_results_df.head(10)

In [ ]:
import pandas as pd
from IPython.display import display

# Temporarily show all rows and all columns for this DataFrame only
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(alpha_results_df)

In [ ]:
alpha_results.to_csv("krippend_simulations.csv")

In [ ]:
scores_piv = scores.pivot_table(
    index=['Query', 'Query Category', 'Method', 'Similarity Score (%)', 'numbered_result', 'Text', 'Psalm Num', 'Verse'],  # or whatever defines a unique row
    columns="User",
    values="Score",
    aggfunc="first"
).reset_index()

scores_piv.iloc[:, 8:]

In [ ]:
import pandas as pd
import itertools
import numpy as np

# df_raters: DataFrame with raters as columns and items as rows
# Example:
# df_raters = pd.DataFrame({
#     'caden': [5, 4, 6],
#     'p01': [5, 3, 6],
#     'p06': [4, 4, 5],
# })

def pairwise_agreement(df):
    raters = df.columns.tolist()
    results = []

    for r1, r2 in itertools.combinations(raters, 2):
        # Only consider items where both raters have a score
        valid = df[[r1, r2]].dropna()
        if len(valid) == 0:
            agreement = np.nan
        else:
            # Compute agreement as proportion of exact matches
            agreement = (valid[r1] == valid[r2]).mean()
        
        results.append({
            'rater_1': r1,
            'rater_2': r2,
            'num_items_compared': len(valid),
            'agreement': agreement
        })

    return pd.DataFrame(results)

# Compute pairwise agreement
pairwise_df = pairwise_agreement(scores_piv.iloc[:, 8:])


# Temporarily show all rows and all columns for this DataFrame only
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(pairwise_df)

In [ ]:
# Sort descending (highest agreement first)
pairwise_df_sorted = pairwise_df.sort_values(by='agreement', ascending=False).reset_index(drop=True)

# Show sorted DataFrame
pairwise_df_sorted.head(15)

In [ ]:

# Suppose avg_filtered contains your filtered users
cols_to_keep = ['p03', 'p17', 'p01', 'p13', 'p07']

# Compute alpha with z-score standardization
alpha_z, df_used_z, users_included_z = compute_krippendorff(
    scores_piv, 
    cols_to_keep, 
    min_ratings_per_item=2, 
    level='interval',  # interval is suitable after z-scoring
    zscore=True
)

print("Users included:", users_included_z)
print("Number of items considered:", df_used_z.shape[0])
print("Krippendorff alpha (z-score standardized):", alpha_z)


## Trying the categorical approach again 

In [ ]:
scores_piv.iloc[:, 8:].columns


In [ ]:
# List of rater columns
raters = ['caden', 'p01', 'p02', 'p03', 'p04', 'p05', 'p06', 
          'p07', 'p08', 'p09', 'p10', 'p13', 'p17']

# Apply score_category and keep missing data
score_catsdf = pd.DataFrame({
    r: scores_piv[r].apply(score_category).astype('Int64') 
    for r in raters
})

# Check results
print(score_catsdf.dtypes)
score_catsdf.head()

In [ ]:
# Convert nullable Int64 DataFrame to float with NaNs
data = score_catsdf.astype(float).to_numpy().T  # rows=raters, columns=items

# Compute Krippendorff alpha for nominal (categorical) data
alpha_nominal = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='nominal'  # 'nominal' for unordered categories
)

print("Krippendorff alpha (categorical/nominal):", alpha_nominal)

In [ ]:
# Convert nullable Int64 DataFrame to float with NaNs
data = score_catsdf.astype(float).to_numpy().T  # rows=raters, columns=items

# Compute Krippendorff alpha for nominal (categorical) data
alpha_nominal = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='ordinal'  # 'nominal' for unordered categories
)

print("Krippendorff alpha (categorical/nominal):", alpha_nominal)

In [ ]:
# Compute the mode for each rater
modes = score_catsdf.mode(dropna=True).iloc[0]

modes_df = modes.reset_index()
modes_df.columns = ['Rater', 'Most_Common_Category']
modes_df

# Sparse Probability of Agreement

In [ ]:
import pandas as pd
import numpy as np

def compute_spa(df, weighting='flat'):
    """
    Compute SPA (Sparse Probability of Agreement) for categorical annotations.
    
    Parameters:
    - df: pd.DataFrame, rows = items, columns = raters, categorical data
    - weighting: 'flat', 'count', or 'variance' (simplest are 'flat' and 'count')
    
    Returns:
    - SPA score (float)
    """
    P_i_list = []
    k_i_list = []
    
    for _, row in df.iterrows():
        counts = row.value_counts(dropna=True)  # counts per category, ignore NaN
        n_i = counts.sum()
        
        if n_i < 2:
            continue  # skip items with <2 ratings
        
        # Probability that two randomly chosen ratings agree for this item
        P_i = sum(counts * (counts - 1)) / (n_i * (n_i - 1))
        P_i_list.append(P_i)
        
        # Weighting
        if weighting == 'flat':
            k_i = 1
        elif weighting == 'count':
            k_i = n_i
        else:
            k_i = 1  # fallback
        
        k_i_list.append(k_i)
    
    # Weighted average
    SPA = np.sum(np.array(P_i_list) * np.array(k_i_list)) / np.sum(k_i_list)
    return SPA

# Example usage: compute SPA for all raters
spa_score = compute_spa(score_catsdf, weighting='count')
print("SPA score:", spa_score)

In [ ]:

spa_score = compute_spa(score_catsdf, weighting='count')
print("SPA score:", spa_score)